In [27]:
import json, numpy as np, pandas as pd
from pathlib import Path
from typing import Optional, List

In [28]:
# 'meta_session', 
# 'vehicle_number', 
# 'meta_time', 
# 'Laptrigger_lapdist_dls', = distance from the start point
# 'Steering_Angle', 
# 'VBOX_Lat_Min', 
# 'VBOX_Long_Minutes', 
# 'accx_can',
# 'accy_can', 'aps', 'gear', 'nmot', 'pbrake_f', 'pbrake_r', 'speed',
       
# 't_s', 
# 't_rel', 'lap', 'sector', 
# 'sector_progress' = the sector progress 0 to 1
# 'run_cum_s', = running time for the entire race
# 'lap_start_s', = current time for lap
# 'lapTime_s' = total time for current lap (s1+s2+s3)
# 't_s1' = time for sec 1 in current lap
# 't_s2' = time for sec 2 in current lap
# 't_s3' = time for sec 3 in current lap
# 'leader_run_cum_s', = the leader current time for current lap
# 'gap_s' = the delta between current car lapTime_s and leader lapTime_s

In [29]:
df = pd.read_parquet('simulation-data/common.parquet')

In [45]:
# df.sort_values(by=['meta_session','vehicle_number','lap','sector','meta_time'], ascending=[True, True, True, True, True])

In [37]:
import numpy as np, pandas as pd

def fill_gaps_long(
    df: pd.DataFrame,
    *,
    sort_keys = ['meta_session','vehicle_number','lap','sector','meta_time'],
    lat_col = None, lon_col = None,          # auto-detected if None
    angular_cols = ('Steering_Angle','heading','Heading','yaw','Yaw','course'),
    target_cols = (),                        # extra metric cols to fill; if empty, auto-detect numeric cols
    time_col = 't_rel',                      # use if present; else fallback to meta_time
    max_gap_s: float | None = None           # don’t interpolate across gaps > this (leave NaN)
) -> pd.DataFrame:
    d = df.copy()

    # choose lat/lon
    if lat_col is None or lon_col is None:
        if {'lat','lon'}.issubset(d.columns): lat_col, lon_col = 'lat','lon'
        elif {'VBOX_Lat_Min','VBOX_Long_Minutes'}.issubset(d.columns): lat_col, lon_col = 'VBOX_Lat_Min','VBOX_Long_Minutes'
        else: raise ValueError("No lat/lon columns found")

    # time axis per row
    if time_col in d.columns:
        d[time_col] = pd.to_numeric(d[time_col], errors='coerce')
        x_name = time_col
    else:
        # build seconds since group start from meta_time
        d['meta_time'] = pd.to_datetime(d['meta_time'], errors='coerce', utc=True)
        x_name = '_t_secs'
        d[x_name] = d.groupby(['meta_session','vehicle_number'])['meta_time'] \
                    .transform(lambda s: (s - s.min()).dt.total_seconds())

    # sort as you prefer
    d = d.sort_values(by=sort_keys, ascending=[True]*len(sort_keys))

    # auto-select target numeric cols if not specified
    if not target_cols:
        numeric_like = d.select_dtypes(include=['number']).columns.tolist()
        # keep core time/keys out
        drop = set(['lap','sector','run_cum_s','lap_start_s','meta_session','vehicle_number', x_name, time_col, 'meta_time'])
        target_cols = [c for c in numeric_like if c not in drop] + [lat_col, lon_col]

        # helpers
    def _interp_linear(x, y):
        x = np.asarray(x, float); y = np.asarray(y, float)
        n = len(x)
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() == 0:
            return y

        out = np.interp(x, x[m], y[m])

        if max_gap_s is not None:
            # nearest known sample index to the left/right (no OOB reads)
            left = np.full(n, -1, dtype=int); last = -1
            for i in range(n):
                if m[i]: last = i
                left[i] = last

            right = np.full(n, n, dtype=int); nxt = n
            for i in range(n - 1, -1, -1):
                if m[i]: nxt = i
                right[i] = nxt

            # build safe neighbor time arrays
            x_left = np.full(n, np.inf)
            maskL = left >= 0
            x_left[maskL] = x[left[maskL]]

            x_right = np.full(n, np.inf)
            maskR = right < n
            x_right[maskR] = x[right[maskR]]

            dist = np.minimum(x - x_left, x_right - x)
            dist[~np.isfinite(dist)] = np.inf  # if both sides inf

            out[dist > max_gap_s] = np.nan

        return out



    def _interp_angle_deg(x, y_deg):
        y = np.deg2rad(y_deg)
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() == 0:
            return y_deg
        yu = np.unwrap(y[m])
        xi = x[m]
        out_u = np.interp(x, xi, yu)
        out = np.rad2deg(out_u)
        return out

    # fill per car group
    keys = ['meta_session','vehicle_number']
    def _fill_group(g: pd.DataFrame) -> pd.DataFrame:
        x = g[x_name].to_numpy()
        for col in target_cols:
            if col not in g.columns: continue
            y_raw = pd.to_numeric(g[col], errors='coerce').to_numpy()
            if col in angular_cols:
                y_new = _interp_angle_deg(x, y_raw)
            else:
                y_new = _interp_linear(x, y_raw)
            # edge hold (so leading/trailing NaNs become nearest valid)
            s = pd.Series(y_new)
            g[col] = s.ffill().bfill().to_numpy()
        return g

    d = d.groupby(keys, group_keys=False).apply(_fill_group)

    # cleanup if we created _t_secs
    if x_name == '_t_secs':
        d = d.drop(columns=['_t_secs'])

    return d


In [38]:
filled = fill_gaps_long(
    df,
    sort_keys=['meta_session','vehicle_number','lap','sector','meta_time'],
    angular_cols=('Steering_Angle'),
    target_cols=(
        'Laptrigger_lapdist_dls','Steering_Angle','accx_can','accy_can','aps','gear',
        'nmot','pbrake_f','pbrake_r','speed','sector_progress','run_cum_s','lap_start_s',
        'lapTime_s','t_s1','t_s2','t_s3','leader_run_cum_s','gap_s',
        'VBOX_Lat_Min','VBOX_Long_Minutes'
    ),
    max_gap_s=3.0   # don’t bridge gaps larger than 3s; set None to disable
)

C:\Users\user\AppData\Local\Temp\ipykernel_18400\2621958107.py:109: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d = d.groupby(keys, group_keys=False).apply(_fill_group)


In [39]:
# Placeholder to fetch the target speed
filled['target'] = 150

In [40]:
filled.to_parquet('simulation-data/common3.parquet')

In [41]:
# export_for_canvas('simulation-data/common.parquet', out_json="race_canvas.json", downsample=2)

In [42]:
# save as export_for_canvas_with_metrics.py
import json, math, numpy as np, pandas as pd
from pathlib import Path
from typing import Optional, List, Dict, Any

# Default telemetry-style columns to try to keep if present
DEFAULT_KEEP_COLS = [
    'Laptrigger_lapdist_dls','Steering_Angle','VBOX_Lat_Min','VBOX_Long_Minutes',
    'accx_can','accy_can','aps','gear','nmot','pbrake_f','pbrake_r','speed',
    't_s','t_rel','lap','sector','sector_progress','run_cum_s','lap_start_s',
    'lapTime_s','t_s1','t_s2','t_s3','leader_run_cum_s','gap_s','target'
]


# DEFAULT_KEEP_COLS = [
#     'Laptrigger_lapdist_dls','Steering_Angle','VBOX_Lat_Min','VBOX_Long_Minutes',
#     'accx_can','accy_can','aps','gear','nmot','pbrake_f','pbrake_r','speed','lap','t_rel'
# ]


def _finite_num(v) -> bool:
    try:
        return math.isfinite(float(v))
    except Exception:
        return False

def _to_list(series: pd.Series) -> List[Any]:
    """JSON-safe: numeric → float (null for NaN), strings otherwise; never NaN."""
    as_num = pd.to_numeric(series, errors="coerce")
    if as_num.notna().mean() >= 0.7:
        return [float(v) if _finite_num(v) else None for v in as_num.tolist()]
    return series.fillna("").astype(str).tolist()

def export_for_canvas(common_path: str,
                      out_json: str = "race_canvas.json",
                      session_filter: Optional[List[str]] = None,
                      downsample: int = 5,
                      keep_cols: Optional[List[str]] = None) -> Dict[str, Any]:
    """
    Build a compact JSON for a race canvas animation with optional extra metrics.
    - Guarantees t/x/y equal-length, finite, and sorted by t.
    - lap_t/lap_v included only if valid and paired.
    - metrics contains only JSON-safe values (no NaN).
    """
    df = pd.read_parquet(common_path) if common_path.endswith(".parquet") else pd.read_csv(common_path)
    #df = df[df['vehicle_number']!='0']
    df = df[~df['vehicle_number'].isin(['0'])]
    #df = df[df['vehicle_number'].isin(['13','21'])]

    
    # required columns
    need = ["meta_session","vehicle_number","t_rel","meta_time"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError(f"missing column(s): {missing}")

    # lat/lon detection (keep your names)
    if "lat" in df.columns and "lon" in df.columns:
        LAT, LON = "lat", "lon"
    elif "VBOX_Lat_Min" in df.columns and "VBOX_Long_Minutes" in df.columns:
        LAT, LON = "VBOX_Lat_Min", "VBOX_Long_Minutes"
    else:
        raise ValueError("No lat/lon found: need either (lat,lon) or (VBOX_Lat_Min,VBOX_Long_Minutes)")

    keep_cols = list(dict.fromkeys((keep_cols or DEFAULT_KEEP_COLS) + ["lap"]))  # ensure 'lap' considered

    df["meta_session"]   = df["meta_session"].astype(str)
    df["vehicle_number"] = df["vehicle_number"].astype(str)
    df = df.dropna(subset=[LAT, LON, "t_rel"]).sort_values(
            ["meta_session","vehicle_number","t_rel"]
        ).reset_index(drop=True)

    if session_filter:
        df = df[df["meta_session"].isin(session_filter)]

    data: Dict[str, Any] = {"sessions": {}}
    step = max(1, int(downsample))

    for sess, g_s in df.groupby("meta_session", sort=False):
        # projection origin per session
        lat_med = g_s[LAT].median()
        lon_med = g_s[LON].median()
        lat0_rad = np.deg2rad(lat_med)
        m_per_deg_lat = 110_540.0
        m_per_deg_lon = 111_320.0 * float(np.cos(lat0_rad))

        def proj_x(series): return (series - lon_med) * m_per_deg_lon
        def proj_y(series): return (series - lat_med) * m_per_deg_lat

        sess_obj = {
            "cars": {},
            "meta": {"lat0": float(lat_med), "lon0": float(lon_med),
                     "proj": "equirectangular_local", "units": {"x":"m","y":"m"}}
        }

        for vid, g in g_s.groupby("vehicle_number", sort=False):
            # Downsample in time order
            h = g.iloc[::step, :].copy()

            # Raw projected coords and time (float64 for filtering; cast later)
            x_raw = proj_x(h[LON].values.astype(float)).astype("float64")
            y_raw = proj_y(h[LAT].values.astype(float)).astype("float64")
            t_raw = pd.to_numeric(h["t_rel"], errors="coerce").astype("float64").values

            # keep only rows with finite t/x/y
            mask = np.isfinite(t_raw) & np.isfinite(x_raw) & np.isfinite(y_raw)
            if not np.any(mask):
                continue  # skip this car entirely

            t = t_raw[mask]
            x = x_raw[mask]
            y = y_raw[mask]
            h = h.iloc[np.nonzero(mask)[0]].reset_index(drop=True)  # align metrics/laps with kept rows

            # enforce ascending t
            order = np.argsort(t)
            t, x, y = t[order], x[order], y[order]
            h = h.iloc[order].reset_index(drop=True)

            car_obj: Dict[str, Any] = {
                "t": t.astype("float32").tolist(),
                "x": x.astype("float32").tolist(),
                "y": y.astype("float32").tolist(),
            }

            # Optional lap arrays: include only if valid and paired
            if "lap" in h.columns:
                laps = pd.to_numeric(h["lap"], errors="coerce").ffill().fillna(0).astype(int).values
                change_idx = np.flatnonzero(np.r_[True, laps[1:] != laps[:-1]])
                if change_idx.size:
                    lt = t[change_idx]
                    lv = laps[change_idx]
                    m2 = np.isfinite(lt)
                    if np.any(m2):
                        car_obj["lap_t"] = lt[m2].astype("float32").tolist()
                        car_obj["lap_v"] = lv[m2].astype("int32").tolist()

            # Extra metrics block (aligned with h; JSON-safe)
            metrics: Dict[str, Any] = {}
            for col in keep_cols:
                if col in h.columns and col not in (LAT, LON, "t_rel"):
                    metrics[col] = _to_list(h[col])
            if metrics:
                car_obj["metrics"] = metrics  # viewer ignores it, but must be valid JSON

            sess_obj["cars"][vid] = car_obj

        data["sessions"][sess] = sess_obj

    Path(out_json).write_text(json.dumps(data), encoding="utf-8")
    print(f"Wrote {out_json}  | sessions={len(data['sessions'])}  cars={sum(len(s['cars']) for s in data['sessions'].values())}")
    return data


In [44]:
test = export_for_canvas('simulation-data/common3.parquet', out_json="race_canvas3.json", downsample=0)

Wrote race_canvas3.json  | sessions=2  cars=34
